In [1]:
import torch
from datasets import load_dataset, Audio
from transformers import WhisperProcessor
from IPython.display import Audio as IPythonAudio, display

# Load the processor just so we can see how it tokenizes text
model_id = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_id, language="Arabic", task="transcribe")

In [2]:
print("Downloading and loading the FLEURS Arabic dataset...")
train_dataset = load_dataset("google/fleurs", "ar_eg", split="train")

# Cast to 16kHz for Whisper
train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16000))

print(f"Total raw training samples: {len(train_dataset)}")

Total raw training samples: 2104


In [3]:
def is_valid_audio(example):
    try:
        _ = example["audio"]["array"]
        return True
    except Exception:
        return False

print("Scanning for corrupted audio...")
clean_train_dataset = train_dataset.filter(is_valid_audio)
print(f"Clean training samples ready: {len(clean_train_dataset)}")

Scanning for corrupted audio...
Clean training samples ready: 2103


In [4]:
# Change this number to explore different audio files in the dataset (0 to 2102)
sample_idx = 1

sample = clean_train_dataset[sample_idx]
audio_array = sample["audio"]["array"]
sampling_rate = sample["audio"]["sampling_rate"]
arabic_text = sample["raw_transcription"]

# Let's also grab the English translation just for context
fleurs_en_train = load_dataset("google/fleurs", "en_us", split="train")
en_translations = {item["id"]: item["raw_transcription"] for item in fleurs_en_train}
english_text = en_translations.get(sample["id"], "Translation not found")

print("="*50)
print(f"Sample ID : {sample['id']}")
print(f"Arabic    : {arabic_text}")
print(f"English   : {english_text}")
print("="*50)

# Display an interactive audio player
display(IPythonAudio(data=audio_array, rate=sampling_rate))

Sample ID : 721
Arabic    : بجانب الشواطئ ذات الرمال البيضاء والمناظر الطبيعية الجبلية، تعد البلاد موطنًا لأقدم مدينة أوروبية في الأمريكتين، وهي الآن جزء من سانتو دومينغو.
English   : Besides white sand beaches and mountain landscapes, the country is home to the oldest European city in the Americas, now part of Santo Domingo.
